# LC-ORF Final Novelty Hardening

Adds prevalence profile, D2 temporal drift diagnostics, explanation-reliability gap, and a final merged audit profile. This notebook does not retrain models and does not run another long bootstrap loop.

In [ ]:
import os, shutil, json, warnings
from pathlib import Path
from datetime import datetime
try:
    from google.colab import drive
    if os.path.isdir('/content/drive') and os.listdir('/content/drive') and not os.path.exists('/content/drive/MyDrive'):
        shutil.rmtree('/content/drive')
    drive.mount('/content/drive', force_remount=True)
except Exception as exc:
    print('Drive mount note:', repr(exc))
    if not os.path.exists('/content/drive/MyDrive'):
        raise
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')
FORMALIZATION_DIR = PROJECT_ROOT / '10_framework_formalization'
RESULTS_DIR = FORMALIZATION_DIR / 'results'
FIGURES_DIR = FORMALIZATION_DIR / 'figures'
LOGS_DIR = FORMALIZATION_DIR / 'logs'
for p in [RESULTS_DIR, FIGURES_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.60, 0.20, 0.20
THRESH = {'average_precision':0.020,'mcc':0.050,'precision':0.050,'recall':0.050,'brier':0.005,'cost_per_10k':100.0,'explanation_instability_index':0.200}
def now(): return datetime.now().isoformat(timespec='seconds')
def log(m):
    line=f'[{now()}] {m}'; print(line)
    with open(LOGS_DIR/'final_novelty_hardening.log','a',encoding='utf-8') as f: f.write(line+'\n')
def write_csv(path, df):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_suffix(path.suffix+'.tmp'); df.to_csv(tmp,index=False); tmp.replace(path)
def write_json(path, obj):
    path=Path(path); tmp=path.with_suffix(path.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2),encoding='utf-8'); tmp.replace(path)
def latest_file(name):
    files=[p for p in PROJECT_ROOT.rglob(name) if p.is_file()]
    if not files: raise FileNotFoundError(name)
    return sorted(files,key=lambda p:p.stat().st_mtime,reverse=True)[0]
def interp(metric, value):
    t=THRESH.get(metric)
    if t is None or pd.isna(value): return 'inconclusive'
    return 'robust' if abs(float(value)) < t else 'fragile'
print(PROJECT_ROOT)

## Prevalence Profile Component

In [ ]:
prev_path = latest_file('prevalence_sensitivity_index.csv')
prev = pd.read_csv(prev_path)
log(f'Loaded {prev_path}')
sampled = prev[prev['sample_status'].eq('sampled')].copy()
metric_map = {'average_precision':'delta_average_precision','precision':'delta_precision','mcc':'delta_mcc','recall':'delta_recall','cost_per_10k':'delta_cost_per_10k','brier':'delta_brier'}
target_rows = []
for (dataset, model, target), g in sampled.groupby(['dataset','model','target_prevalence'], dropna=False):
    for metric, col in metric_map.items():
        vals = g[col].dropna().astype(float)
        if len(vals) == 0: continue
        target_rows.append({'audit_axis':'prevalence','dataset':dataset,'model':model,'target_prevalence':float(target),'metric':metric,'n_rows':len(vals),'n_seeds':int(g['seed'].nunique()),'delta_mean':float(vals.mean()),'delta_std':float(vals.std(ddof=1)) if len(vals)>1 else 0.0,'delta_min':float(vals.min()),'delta_max':float(vals.max()),'max_abs_delta':float(vals.abs().max()),'interpretation':interp(metric, vals.abs().max())})
prev_target = pd.DataFrame(target_rows)
write_csv(RESULTS_DIR/'prevalence_profile_per_target.csv', prev_target)
comp = []
for (dataset, model, metric), g in prev_target.groupby(['dataset','model','metric'], dropna=False):
    worst = g.sort_values('max_abs_delta', ascending=False).iloc[0]
    comp.append({'audit_axis':'prevalence','dataset':dataset,'model':model,'reference_scenario':'NATURAL_TEST_POOL','perturbed_scenario':'TARGET_PREVALENCE_RANGE','changed_assumption':'class prevalence in evaluation stream','metric':metric,'artifact_id':'M07-PSI','n_seeds':int(g['n_seeds'].max()),'delta_mean_across_targets':float(g['delta_mean'].mean()),'worst_target_prevalence':float(worst['target_prevalence']),'max_abs_delta':float(worst['max_abs_delta']),'interpretation':worst['interpretation'],'evidence_status':'complete'})
prevalence_component = pd.DataFrame(comp)
write_csv(RESULTS_DIR/'prevalence_profile_component.csv', prevalence_component)
display(prevalence_component.head(20))

## D2 Temporal Drift Diagnostics

In [ ]:
def find_dataset_file(file_name):
    roots=[PROJECT_ROOT/'Datasets', PROJECT_ROOT/'00_source_datasets', PROJECT_ROOT]
    for r in roots:
        p=r/file_name
        if p.exists(): return p
        if r.exists():
            hits=list(r.glob(f'**/{file_name}'))
            if hits: return hits[0]
    raise FileNotFoundError(file_name)
def psi(a,b,bins=10):
    a=pd.Series(a).dropna().astype(float); b=pd.Series(b).dropna().astype(float)
    edges=np.unique(np.nanquantile(a, np.linspace(0,1,bins+1)))
    if len(edges)<3: edges=np.linspace(min(a.min(),b.min()), max(a.max(),b.max()), bins+1)
    edges[0], edges[-1] = -np.inf, np.inf
    ac,_=np.histogram(a,bins=edges); bc,_=np.histogram(b,bins=edges)
    ap=np.maximum(ac/max(ac.sum(),1),1e-6); bp=np.maximum(bc/max(bc.sum(),1),1e-6)
    return float(np.sum((bp-ap)*np.log(bp/ap)))
def jsd(a,b,bins=20):
    a=pd.Series(a).dropna().astype(float); b=pd.Series(b).dropna().astype(float)
    if len(a)==0 or len(b)==0: return np.nan
    lo,hi=min(a.min(),b.min()),max(a.max(),b.max())
    if lo==hi: return 0.0
    edges=np.linspace(lo,hi,bins+1); ac,_=np.histogram(a,bins=edges); bc,_=np.histogram(b,bins=edges)
    ap=np.maximum(ac/max(ac.sum(),1),1e-12); bp=np.maximum(bc/max(bc.sum(),1),1e-12)
    return float(jensenshannon(ap,bp,base=2)**2)
def bh(p):
    p=np.asarray(p,float); order=np.argsort(p); out=np.empty_like(p); prev=1.0; n=len(p)
    for i in range(n-1,-1,-1):
        idx=order[i]; val=min(prev,p[idx]*n/(i+1)); out[idx]=val; prev=val
    return np.clip(out,0,1)
d2_path=find_dataset_file('fraudTest.csv')
log(f'Loading D2 from {d2_path}')
d2=pd.read_csv(d2_path, low_memory=False)
d2['__order__']=pd.to_datetime(d2['trans_date_trans_time'], errors='coerce')
d2=d2.dropna(subset=['__order__','is_fraud']).sort_values('__order__').reset_index(drop=True)
d2['is_fraud']=d2['is_fraud'].astype(int)
d2['trans_dt']=pd.to_datetime(d2['trans_date_trans_time'], errors='coerce')
d2['dob_dt']=pd.to_datetime(d2['dob'], errors='coerce')
d2['trans_hour']=d2['trans_dt'].dt.hour; d2['trans_dayofweek']=d2['trans_dt'].dt.dayofweek; d2['trans_month']=d2['trans_dt'].dt.month
d2['age_years']=(d2['trans_dt']-d2['dob_dt']).dt.days/365.25
n=len(d2); train=d2.iloc[:int(n*TRAIN_FRAC)]; val=d2.iloc[int(n*TRAIN_FRAC):int(n*(TRAIN_FRAC+VAL_FRAC))]; test=d2.iloc[int(n*(TRAIN_FRAC+VAL_FRAC)):]
features=['amt','city_pop','trans_hour','trans_dayofweek','trans_month','age_years']
rows=[]
for feat in features:
    a=pd.to_numeric(train[feat],errors='coerce'); b=pd.to_numeric(test[feat],errors='coerce'); ks=ks_2samp(a.dropna(),b.dropna())
    rows.append({'dataset':'D2_fraudTest','feature':feat,'train_n':int(a.notna().sum()),'test_n':int(b.notna().sum()),'train_mean':float(a.mean()),'test_mean':float(b.mean()),'train_median':float(a.median()),'test_median':float(b.median()),'psi':psi(a,b),'js_divergence':jsd(a,b),'ks_statistic':float(ks.statistic),'ks_p_value':float(ks.pvalue)})
drift=pd.DataFrame(rows); drift['ks_p_value_bh']=bh(drift['ks_p_value']); drift['drift_flag']=np.select([drift.psi>=0.25, drift.psi>=0.10], ['large','moderate'], default='small')
splits=pd.DataFrame([{'dataset':'D2_fraudTest','split':name,'n':len(part),'fraud_count':int(part.is_fraud.sum()),'fraud_rate':float(part.is_fraud.mean()),'start_time':str(part.__order__.min()),'end_time':str(part.__order__.max())} for name,part in [('train',train),('validation',val),('chronological_test',test)]])
temporal=pd.read_csv(latest_file('temporal_robustness_index.csv'))
drop=temporal[temporal.dataset.eq('D2_fraudTest')].groupby(['dataset','model']).agg(ap_drop_mean=('trd_average_precision','mean'),ap_drop_std=('trd_average_precision','std'),mcc_drop_mean=('trd_mcc','mean'),mcc_drop_std=('trd_mcc','std'),n_seeds=('seed','nunique')).reset_index()
write_csv(RESULTS_DIR/'d2_temporal_drift_diagnostics.csv', drift)
write_csv(RESULTS_DIR/'d2_temporal_prevalence_by_split.csv', splits)
write_csv(RESULTS_DIR/'d2_temporal_drop_summary.csv', drop)
display(drift.sort_values('psi',ascending=False)); display(splits); display(drop)

## Explanation Reliability Gap

In [ ]:
protocol=pd.read_csv(latest_file('protocol_explanation_stability_index.csv'))
protocol5=protocol[protocol.top_k.eq(5)].copy()
boot=pd.read_csv(latest_file('paired_bootstrap_delta_summary.csv'))
perf=boot[(boot.audit_axis.eq('explanation_protocol')) & (boot.metric.isin(['average_precision','mcc']))]
perf_wide=perf.pivot_table(index=['dataset','model'],columns='metric',values='delta_mean_across_seeds',aggfunc='mean').reset_index().rename(columns={'average_precision':'protocol_ap_delta_mean','mcc':'protocol_mcc_delta_mean'})
exp=protocol5.groupby(['dataset','model']).agg(top5_jaccard_mean=('topk_jaccard','mean'),top5_jaccard_std=('topk_jaccard','std'),eii_mean=('explanation_instability_index','mean'),eii_std=('explanation_instability_index','std'),importance_spearman_mean=('importance_spearman','mean'),importance_spearman_std=('importance_spearman','std'),n_seeds=('seed','nunique')).reset_index()
gap=exp.merge(perf_wide,on=['dataset','model'],how='left')
gap['ap_fragility_units']=gap.protocol_ap_delta_mean.abs()/THRESH['average_precision']
gap['mcc_fragility_units']=gap.protocol_mcc_delta_mean.abs()/THRESH['mcc']
gap['explanation_reliability_gap']=gap.ap_fragility_units-gap.eii_mean
gap['gap_interpretation']=np.select([gap.explanation_reliability_gap>1.0, gap.eii_mean>=THRESH['explanation_instability_index'], gap.ap_fragility_units<1.0], ['predictive_shift_exceeds_rank_instability','explanation_rank_instability_detected','predictive_shift_not_material'], default='mixed')
gap['artifact_id']='M09-EII-GAP'
write_csv(RESULTS_DIR/'explanation_reliability_gap.csv', gap)
display(gap)

## Final Audit Profile Merge

In [ ]:
base=pd.read_csv(latest_file('lcorf_audit_profile.csv'))
prev_prof=prevalence_component.rename(columns={'delta_mean_across_targets':'delta_mean_across_seeds','max_abs_delta':'delta_display'}).copy()
prev_prof['profile_component']='prevalence:'+prev_prof.metric.astype(str)
prev_prof['ci_low']=np.nan; prev_prof['ci_high']=np.nan; prev_prof['n_bootstrap_min']=np.nan; prev_prof['bootstrap_modes']='deterministic_resampling_summary'
prev_prof['interpretation_counts']=prev_prof.interpretation.apply(lambda x: json.dumps({x:1}))
gap_rows=[]
for _,r in gap.iterrows():
    gap_rows.append({'audit_axis':'explanation_reliability_gap','dataset':r.dataset,'model':r.model,'reference_scenario':'EXPLANATION_STABILITY','perturbed_scenario':'PREDICTIVE_PROTOCOL_SHIFT','changed_assumption':'agreement between explanation-rank stability and predictive protocol fragility','metric':'explanation_reliability_gap','artifact_id':r.artifact_id,'n_seeds':r.n_seeds,'delta_mean_across_seeds':r.explanation_reliability_gap,'delta_std_across_seeds':np.nan,'ci_low_median':np.nan,'ci_high_median':np.nan,'n_bootstrap_min':np.nan,'bootstrap_modes':'artifact_join','interpretation_counts':json.dumps({r.gap_interpretation:1}),'profile_component':'explanation_reliability_gap','delta_display':r.explanation_reliability_gap,'ci_low':np.nan,'ci_high':np.nan,'evidence_status':'complete_proxy_from_existing_artifacts'})
gap_prof=pd.DataFrame(gap_rows)
final=pd.concat([base, prev_prof.reindex(columns=base.columns), gap_prof.reindex(columns=base.columns)], ignore_index=True)
write_csv(RESULTS_DIR/'lcorf_final_audit_profile.csv', final)
print('base',len(base),'prevalence',len(prev_prof),'gap',len(gap_prof),'final',len(final))
display(final.tail(20))

## Figures

In [ ]:
prev_heat=prevalence_component[prevalence_component.metric.isin(['average_precision','precision','mcc','recall','cost_per_10k'])].copy()
prev_heat['row']=prev_heat.dataset+' / '+prev_heat.model
tab=prev_heat.pivot_table(index='row',columns='metric',values='max_abs_delta',aggfunc='mean')
plt.figure(figsize=(8.5,max(3.5,0.45*len(tab))))
sns.heatmap(tab,cmap='mako',annot=True,fmt='.3f',linewidths=.4)
plt.title('Prevalence Stress Profile: Maximum Absolute Delta'); plt.xlabel(''); plt.ylabel(''); plt.tight_layout()
plt.savefig(FIGURES_DIR/'prevalence_profile_heatmap.png',dpi=600,bbox_inches='tight'); plt.savefig(FIGURES_DIR/'prevalence_profile_heatmap.pdf',bbox_inches='tight'); plt.show()

# Journal/conference-ready replacement for the old PSI-only D2 figure.
# Same output filenames are preserved, so Google Drive will overwrite the previous figure.
fig,axes=plt.subplots(1,3,figsize=(14.6,4.4),gridspec_kw={'width_ratios':[1.0,1.35,1.15]})

split_order=['train','validation','chronological_test']
if 'd2_prev' in globals():
    prev_plot=d2_prev.copy()
elif 'splits' in globals():
    prev_plot=splits.copy()
else:
    prev_plot=pd.read_csv(RESULTS_DIR/'d2_temporal_prevalence_by_split.csv')
prev_plot['split']=pd.Categorical(prev_plot['split'],categories=split_order,ordered=True)
prev_plot=prev_plot.sort_values('split')
sns.barplot(data=prev_plot,x='split',y='fraud_rate',ax=axes[0],color='#4C78A8',errorbar=None)
axes[0].set_title('D2 Fraud Rate by Time Split')
axes[0].set_xlabel('')
axes[0].set_ylabel('Fraud rate')
axes[0].set_xticklabels(['Train','Validation','Chronological\ntest'])
for p in axes[0].patches:
    h=p.get_height()
    axes[0].annotate(f'{h:.3%}',(p.get_x()+p.get_width()/2,h),ha='center',va='bottom',fontsize=8,xytext=(0,3),textcoords='offset points')

metric_labels={'ap_drop_mean':'Average precision','mcc_drop_mean':'MCC'}
drop_plot=drop.melt(id_vars=['dataset','model'],value_vars=['ap_drop_mean','mcc_drop_mean'],var_name='metric',value_name='mean_delta')
drop_plot['metric']=drop_plot['metric'].map(metric_labels)
sns.barplot(data=drop_plot,x='metric',y='mean_delta',hue='model',ax=axes[1],palette=['#4C78A8','#F58518'],errorbar=None)
axes[1].axhline(0,color='black',linewidth=.8)
axes[1].set_title('D2 Chronological Performance Delta')
axes[1].set_xlabel('')
axes[1].set_ylabel('Chronological - random')
axes[1].legend(title='Model',frameon=True,loc='lower right')
for container in axes[1].containers:
    axes[1].bar_label(container,fmt='%.3f',fontsize=8,padding=2)

# Keep drift evidence, but cap the dominant month effect so smaller feature shifts remain visible.
drift_plot=drift.sort_values('psi',ascending=False).copy()
drift_plot['psi_display']=drift_plot['psi'].clip(upper=0.30)
drift_plot['label']=drift_plot.apply(lambda r: f"{r['feature']} ({r['psi']:.3f})" if r['psi']>0.30 else r['feature'],axis=1)
sns.barplot(data=drift_plot,x='psi_display',y='label',ax=axes[2],color='#72B7B2',errorbar=None)
axes[2].axvline(0.10,color='#B279A2',linestyle='--',linewidth=1,label='PSI=0.10')
axes[2].set_title('D2 Feature Drift (PSI, capped)')
axes[2].set_xlabel('PSI display scale')
axes[2].set_ylabel('')
axes[2].legend(frameon=True,loc='lower right',fontsize=8)
axes[2].text(0.02,-0.16,'Values above 0.30 are labeled with true PSI.',transform=axes[2].transAxes,fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR/'d2_drift_vs_temporal_drop.png',dpi=600,bbox_inches='tight')
plt.savefig(FIGURES_DIR/'d2_drift_vs_temporal_drop.pdf',bbox_inches='tight')
plt.show()

plt.figure(figsize=(8,4.8))
sns.barplot(data=gap,x='dataset',y='explanation_reliability_gap',hue='model')
plt.axhline(0,color='black',linewidth=.8); plt.title('Explanation Reliability Gap'); plt.ylabel('AP fragility units - explanation instability'); plt.tight_layout()
plt.savefig(FIGURES_DIR/'explanation_reliability_gap.png',dpi=600,bbox_inches='tight'); plt.savefig(FIGURES_DIR/'explanation_reliability_gap.pdf',bbox_inches='tight'); plt.show()


## Summary Checks

In [ ]:
summary={'completed_at':now(),'prevalence_profile_component':str(RESULTS_DIR/'prevalence_profile_component.csv'),'d2_temporal_drift_diagnostics':str(RESULTS_DIR/'d2_temporal_drift_diagnostics.csv'),'explanation_reliability_gap':str(RESULTS_DIR/'explanation_reliability_gap.csv'),'final_audit_profile':str(RESULTS_DIR/'lcorf_final_audit_profile.csv'),'figures_dir':str(FIGURES_DIR),'final_profile_rows':int(len(final))}
write_json(RESULTS_DIR/'lcorf_final_novelty_hardening_run_summary.json', summary)
print(json.dumps(summary,indent=2))
assert len(prevalence_component)>0
assert len(drift)>0
assert len(gap)>0
assert (RESULTS_DIR/'lcorf_final_audit_profile.csv').exists()